# Onechess : essai d'entraînement sur Colab

Essai **R1024HS** : réseau 1024, 4 experts par matériel, couches cachées 16/32 à produit par paires, **parcimonieuses** (`--act-l1 0.005`), sur ~170 M de positions de l'autojeu public de Stockfish (`official-stockfish/master-binpacks`, `nodes5000pv2_UHO`, licence ODbL).

**À faire une fois :** *Exécution → Modifier le type d'exécution → GPU T4*, puis *Exécution → Tout exécuter*, et laisser l'onglet ouvert. Colab demande l'accès à Google Drive : la progression y est sauvée (`MyDrive/onechess/`), et chaque nouvelle ouverture reprend où l'entraînement s'était arrêté. Aucune clé ni mot de passe n'est nécessaire.

In [ ]:
# 1. GPU et Google Drive (la progression est sauvee dans MyDrive/onechess)
import subprocess, os
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount('/content/drive')
SORTIE = '/content/drive/MyDrive/onechess'
os.makedirs(SORTIE, exist_ok=True)
print('sauvegarde :', SORTIE, os.listdir(SORTIE))

In [ ]:
# 2. Code d'Onechess (depot public) et donnees publiques, converties en 32 octets par position.
# Deterministe : refaite a chaque session, elle redonne les memes fichiers, donc la reprise est exacte.
import glob, io, tarfile, urllib.request, sys, time, numpy as np
t0 = time.time()
tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen('https://codeload.github.com/ovnice-dev/onechess/tar.gz/main').read())).extractall('/content/code')
TOOLS = glob.glob('/content/code/*/tools')[0]
if not glob.glob('/content/donnees/*.bin'):
    subprocess.run([sys.executable, '-u', TOOLS + '/hf_sources.py',
                    '--url', 'https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/nodes5000pv2_UHO.binpack',
                    '--prefixe', 'n5', '--dest', '/content/donnees', '--octets', str(2_000_000_000),
                    '--garde', '1.0', '--fixe', '240', '--graine', '21', '--parts', str(os.cpu_count())], check=True)
    v = sorted(glob.glob('/content/donnees/*_p00_*.bin'))[0]
    np.fromfile(v, dtype='uint8').reshape(-1, 32)[:5_000_000].tofile('/content/donnees/z_val.bin')
    os.remove(v)
n = sum(os.path.getsize(f) for f in glob.glob('/content/donnees/*.bin')) // 32
print(f'{n:,} positions, {len(glob.glob("/content/donnees/*.bin"))} fichiers, {time.time()-t0:.0f} s')

In [ ]:
# 3. Entrainement (20 epoques ; point de reprise a chaque epoque sur Drive). Relancer cette cellule reprend.
cmd = [sys.executable, '-u', TOOLS + '/train.py', '--stream', '/content/donnees/*.bin', '--chunk-files', '10',
       '--hidden', '1024', '--l1', '16', '--l2', '32', '--pairmul', '1', '--act-l1', '0.005',
       '--buckets', 'mat4', '--out-buckets', '4', '--epochs', '20',
       '--checkpoint', SORTIE + '/r1024hs.ckpt', '--out', SORTIE + '/r1024hs.nnue']
print(' '.join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(SORTIE + '/r1024hs.log', 'a') as log:
    for line in p.stdout:
        print(line, end='')
        log.write(line); log.flush()
print('code', p.wait())

Quand la ligne `reseau ecrit` apparaît, le réseau est dans `MyDrive/onechess/r1024hs.nnue`. Il sera ensuite vérifié (`check_nnue`), réordonné (`ftperm`) et jugé par un match au temps contre R1024.